# Data Audit — UCI Bank Marketing

01_eda.py — Data audit for the UCI Bank Marketing dataset.

REFERENCE IMPLEMENTATION — read this to understand the dataset, then be
ready to explain every number and every exclusion decision yourself.
This file answers questions; it does not make modelling decisions.

Decision point for the whole project: the bank chooses whom to call
BEFORE the call happens. So for every column we ask: "would the bank
know this value at the moment it decides to dial the number?"

In [15]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 1. Load

In [16]:
df = pd.read_csv("data/bank-additional-full.csv", sep=";")

print("SHAPE")
print(df.shape)

print("\n")
print("DTYPES")
print(df.dtypes)

SHAPE
(41188, 21)


DTYPES
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object


## 2. Target balance

In [17]:
print("TARGET BALANCE (y)")
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True).round(4))
# This is a highly imbalanced problem (~11% positive class). That single
# fact rules out accuracy as a usable metric: predicting "no" for every
# row already scores ~89% accuracy while being useless to the bank.

TARGET BALANCE (y)
y
no     36548
yes     4640
Name: count, dtype: int64
y
no     0.8873
yes    0.1127
Name: proportion, dtype: float64


## 3. Categorical vs numeric fields

In [18]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("y")
numeric_cols = df.select_dtypes(include="number").columns.tolist()

print("CATEGORICAL COLUMNS")
print(categorical_cols)

print("\n")
print("NUMERIC COLUMNS")
print(numeric_cols)

CATEGORICAL COLUMNS
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


NUMERIC COLUMNS
['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


C:\Users\daksh\AppData\Local\Temp\ipykernel_26448\4070335129.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


## 4. "unknown" values — how much of each categorical column is unknown?

In [19]:
print("'unknown' RATE PER CATEGORICAL COLUMN")
unknown_rates = (
    df[categorical_cols]
    .apply(lambda s: (s == "unknown").mean())
    .sort_values(ascending=False)
)
print(unknown_rates[unknown_rates > 0].round(4))
# default has a very high unknown rate — worth noting explicitly, since a
# column that's mostly "unknown" carries little information no matter how
# you encode it.

'unknown' RATE PER CATEGORICAL COLUMN
default      0.2087
education    0.0420
housing      0.0240
loan         0.0240
job          0.0080
marital      0.0019
dtype: float64


## 5. Special numeric codes

In [20]:
print("pdays SPECIAL CODE")
print("pdays == 999 (never previously contacted):",
      (df["pdays"] == 999).mean().round(4), "of rows")
print("pdays distribution for the remaining rows:")
print(df.loc[df["pdays"] != 999, "pdays"].describe())
# 999 is a sentinel, not a real day count. Leaving it untransformed and
# feeding it into a model that assumes continuous scale (e.g. LDA/QDA,
# logistic regression) would make "never contacted" look like
# "contacted 999 days ago" — wrong. This is a DECISION POINT: common
# treatments are (a) a binary "previously_contacted" flag + the real
# pdays for the rest, (b) leave as-is and accept the distortion, or
# (c) bucket pdays. Pick one and be ready to say why.

print("\n")
print("previous / poutcome cross-check")
print(pd.crosstab(df["previous"] == 0, df["poutcome"]))
# Confirms poutcome == 'nonexistent' lines up with previous == 0 and
# pdays == 999 — these three columns tell a consistent story about
# clients never contacted before.

pdays SPECIAL CODE
pdays == 999 (never previously contacted): 0.9632 of rows
pdays distribution for the remaining rows:
count    1515.000000
mean        6.014521
std         3.824906
min         0.000000
25%         3.000000
50%         6.000000
75%         7.000000
max        27.000000
Name: pdays, dtype: float64


previous / poutcome cross-check
poutcome  failure  nonexistent  success
previous                               
False        4252            0     1373
True            0        35563        0


## 6. Feature availability audit — THE key decision for this project

In [21]:
print("FEATURE AVAILABILITY AT DECISION POINT")

availability = {
    "age": "available (client attribute)",
    "job": "available (client attribute)",
    "marital": "available (client attribute)",
    "education": "available (client attribute)",
    "default": "available, but mostly 'unknown' — low info content",
    "housing": "available (client attribute)",
    "loan": "available (client attribute)",
    "contact": "available (contact channel is chosen before the call)",
    "month": "available (call is scheduled before it happens)",
    "day_of_week": "available (call is scheduled before it happens)",
    "duration": "NOT AVAILABLE — call duration is only known AFTER the "
                "call ends. UCI's own documentation flags this column as "
                "leakage for realistic prediction. MUST BE EXCLUDED.",
    "campaign": "available (running count of contacts so far this "
                "campaign for this client, known before dialling again)",
    "pdays": "available (outcome of a PAST campaign, known in advance)",
    "previous": "available (outcome of a PAST campaign, known in advance)",
    "poutcome": "available (outcome of a PAST campaign, known in advance)",
    "emp.var.rate": "available (published macro indicator) but reflects "
                    "TIME PERIOD more than the client — correlates with "
                    "which month the row is from",
    "cons.price.idx": "same caveat as emp.var.rate",
    "cons.conf.idx": "same caveat as emp.var.rate",
    "euribor3m": "same caveat as emp.var.rate",
    "nr.employed": "same caveat as emp.var.rate",
}
for col, note in availability.items():
    print(f"  {col:16s} -> {note}")

print(
    "\nEXCLUSION: 'duration' must be dropped before any supervised "
    "modelling — it is the single clearest leakage risk in this dataset.\n"
    "WATCH: the five economic-indicator columns are technically available "
    "pre-call, but because the dataset is date-ordered, they're highly "
    "correlated with WHEN a row was collected, not just general macro "
    "conditions. With a temporal train/holdout split this can make the "
    "model partly a 'which time period is this' detector rather than a "
    "'which client is promising' detector. Keep them in but be ready to "
    "flag this as a limitation — don't silently ignore it."
)

FEATURE AVAILABILITY AT DECISION POINT
  age              -> available (client attribute)
  job              -> available (client attribute)
  marital          -> available (client attribute)
  education        -> available (client attribute)
  default          -> available, but mostly 'unknown' — low info content
  housing          -> available (client attribute)
  loan             -> available (client attribute)
  contact          -> available (contact channel is chosen before the call)
  month            -> available (call is scheduled before it happens)
  day_of_week      -> available (call is scheduled before it happens)
  duration         -> NOT AVAILABLE — call duration is only known AFTER the call ends. UCI's own documentation flags this column as leakage for realistic prediction. MUST BE EXCLUDED.
  campaign         -> available (running count of contacts so far this campaign for this client, known before dialling again)
  pdays            -> available (outcome of a PAST campa